# 02 - Create and verify the delivery label

**Job:** compare actual customer delivery date with the promised
estimated date, verify real examples, and measure class imbalance.

**Input:** `artifacts/01_ml_table.csv.gz`

**Output:** `artifacts/02_labeled_table.csv.gz`

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
ARTIFACT_DIR = ROOT / "artifacts"
input_path = ARTIFACT_DIR / "01_ml_table.csv.gz"
assert input_path.exists(), "Run Notebook 01 first."

ml_table = pd.read_csv(input_path, low_memory=False)
DATE_COLUMNS = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date", "shipping_limit_date_max",
]
for column in DATE_COLUMNS:
    if column in ml_table:
        ml_table[column] = pd.to_datetime(ml_table[column], errors="coerce")
print(f"Input shape: {ml_table.shape}")

Input shape: (99441, 41)


## Label rule

Only completed deliveries with both timestamps are labelable. Task 1's
SQL compares the complete timestamps, so this notebook preserves that
exact baseline definition: `late = 1` when actual delivery is later than
the estimated timestamp. Because estimates are stored at midnight, a
same-date delivery after midnight is late under the Task 1 rule.

In [2]:
labeled = ml_table.loc[
    ml_table["order_status"].eq("delivered")
    & ml_table["order_delivered_customer_date"].notna()
    & ml_table["order_estimated_delivery_date"].notna()
].copy()

actual_timestamp = labeled["order_delivered_customer_date"]
estimated_timestamp = labeled["order_estimated_delivery_date"]
labeled["delivery_delay_days"] = (
    actual_timestamp - estimated_timestamp
).dt.total_seconds() / 86400
labeled["late"] = actual_timestamp.gt(estimated_timestamp).astype("int8")
labeled["delivery_label"] = np.where(labeled["late"].eq(1), "late", "on_time")

assert labeled["order_id"].is_unique
assert labeled["late"].isin([0, 1]).all()
assert labeled["late"].eq(labeled["delivery_delay_days"].gt(0).astype("int8")).all()
print(f"Labelable delivered orders: {len(labeled):,}")

Labelable delivered orders: 96,470


## Check the rule on real orders

In [3]:
verification_columns = [
    "order_id", "order_delivered_customer_date",
    "order_estimated_delivery_date", "delivery_delay_days",
    "late", "delivery_label",
]
examples = pd.concat([
    labeled.loc[labeled["late"].eq(1), verification_columns].head(3),
    labeled.loc[labeled["late"].eq(0), verification_columns].head(3),
], ignore_index=True)
examples["manual_rule_check"] = np.where(
    examples["order_delivered_customer_date"]
    > examples["order_estimated_delivery_date"],
    "late", "on_time",
)
assert examples["manual_rule_check"].eq(examples["delivery_label"]).all()
display(examples)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,late,delivery_label,manual_rule_check
0,203096f03d82e0dffbc41ebc2e2bcfb7,2017-10-09 22:23:46,2017-09-28,11.933171,1,late,late
1,fbf9ac61453ac646ce8ad9783d7d0af6,2018-03-21 22:03:54,2018-03-12,9.919375,1,late,late
2,8563039e855156e48fccee4d611a3196,2018-03-20 00:59:25,2018-03-20,0.041262,1,late,late
3,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,-7.107488,0,on_time,on_time
4,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,-5.355729,0,on_time,on_time
5,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,-17.245498,0,on_time,on_time


## Class distribution and imbalance decision

In [4]:
distribution = (
    labeled["delivery_label"].value_counts()
    .rename_axis("delivery_label").reset_index(name="orders")
)
distribution["percentage"] = 100 * distribution["orders"] / len(labeled)
display(distribution)

majority = int(distribution["orders"].max())
minority = int(distribution["orders"].min())
imbalance_ratio = majority / minority
imbalance_level = "severe" if imbalance_ratio >= 4 else ("moderate" if imbalance_ratio >= 2 else "mild")
print(
    f"Decision: {imbalance_level} class imbalance ({imbalance_ratio:.2f}:1). "
    "Use stratification for random splits if selected and evaluate PR-AUC, recall, "
    "precision, F1, and balanced accuracy rather than accuracy alone."
)

,delivery_label,orders,percentage
0,on_time,88644,91.887633
1,late,7826,8.112367


Decision: severe class imbalance (11.33:1). Use stratification for random splits if selected and evaluate PR-AUC, recall, precision, F1, and balanced accuracy rather than accuracy alone.


## Save the step-2 artifacts

In [5]:
output_path = ARTIFACT_DIR / "02_labeled_table.csv.gz"
labeled.to_csv(output_path, index=False, compression="gzip")
distribution.to_csv(ARTIFACT_DIR / "02_label_distribution.csv", index=False)
examples.to_csv(ARTIFACT_DIR / "02_label_verification_examples.csv", index=False)
summary = {
    "label_rule": "late=1 when delivered timestamp > estimated timestamp (Task 1 baseline)",
    "baseline_consistency_check": "matches Task 1 SQL timestamp comparison",
    "labelable_rows": int(len(labeled)),
    "late_rows": int(labeled["late"].sum()),
    "on_time_rows": int(labeled["late"].eq(0).sum()),
    "late_rate": float(labeled["late"].mean()),
    "majority_to_minority_ratio": float(imbalance_ratio),
    "imbalance_level": imbalance_level,
}
(ARTIFACT_DIR / "02_label_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
assert output_path.exists() and output_path.stat().st_size > 0
print(f"Saved {output_path.relative_to(ROOT)}")

Saved artifacts\02_labeled_table.csv.gz
